# AInstein component: LLM QA

In [1]:
import sys
from pathlib import Path

path_project = Path.cwd().parent
sys.path.append(str(path_project))

In [2]:
from AInstein import (
    get_llm_azure_openai, # conexión al modelo GPT
    get_settings, # conexión a las configuraciones de los modelos
    BigQueryManager
)

In [3]:
import re
import json
from collections import defaultdict

import ipywidgets as widgets
import pandas as pd
from IPython.display import display, Markdown, HTML
from openpyxl import load_workbook
from io import BytesIO
import copy

# Settings

In [5]:
# Settings
ENVIRONMENT: str = 'bdb-gcp-sbx-ia'
WORKPLACE_PROJECT_ID: str = 'geo-cargas_laborales'

# Obtener las configuraciones del proyecto
settings = get_settings(WORKPLACE_PROJECT_ID, environment=ENVIRONMENT)

# Models

In [6]:
# Crear instancias de los modelos
llm = get_llm_azure_openai(settings)  # Modelo de Azure OpenAI

# Responses

## General

In [8]:
import re
import json
import copy
from io import BytesIO
from openpyxl import load_workbook

# ===========================================================================
#  ProcesadorTranscripcionTeams
#  v4 — flujo conversacional mejorado + Excel de salida + BigQuery
# ===========================================================================

class ProcesadorTranscripcionTeams:
    """
    Procesa archivos VTT de Microsoft Teams para levantar actividades,
    enriquecerlas con IA y generar resúmenes confirmables.

    CAMBIOS v4
    ----------
    1. El análisis de carga POR ACTIVIDAD se calcula internamente pero
       ya NO se muestra al usuario (queda reservado para el analista).
       El LLM usa las métricas como contexto silencioso para validar
       coherencia, pero el usuario solo ve el resumen de sus datos.
    2. La autonomía ya NO se pregunta como porcentaje: se pregunta
       cuántas personas realizan la actividad y el sistema calcula
       automáticamente el porcentaje con la fórmula 1/n × 100.
    3. La pregunta de proceso de área se reformuló: ya no dice
       "proceso o área" sino solo "proceso del área".
    4. Al inicio del pipeline se solicita cargo y vicepresidencia de
       forma conversacional (sin widgets) antes de capturar actividades.
    5. Al finalizar se genera un Excel descargable con los datos del
       usuario sobre el formato estándar de Levantamiento de Cargas.
    6. Los datos finales se insertan en una tabla de BigQuery.
    """

    # -----------------------------------------------------------------------
    # CONSTANTES COMPARTIDAS
    # -----------------------------------------------------------------------
    FRECUENCIAS_VALIDAS = [
        "Diario", "Semanal", "Quincenal",
        "Mensual", "Bimensual", "Trimestral", "Semestral", "Anual"
    ]

    # -----------------------------------------------------------------------
    # 1. LECTURA Y LIMPIEZA DEL VTT
    # -----------------------------------------------------------------------
    def leer_archivo_vtt(self, contenido):
        conversaciones = []
        lineas = contenido.splitlines()

        timestamp_actual = None
        buffer_texto = []

        for linea in lineas:
            linea = linea.strip()

            if (
                not linea
                or linea == "WEBVTT"
                or re.match(r"^[a-f0-9\-]+\/\d+\-\d+$", linea)
            ):
                continue

            if "-->" in linea:
                if timestamp_actual and buffer_texto:
                    conversaciones.append({
                        "timestamp": timestamp_actual,
                        "texto": " ".join(buffer_texto).strip()
                    })
                    buffer_texto = []

                inicio = linea.split("-->")[0].strip()
                timestamp_actual = inicio.split(".")[0]
                continue

            linea = re.sub(r"<v[^>]*>", "", linea)
            linea = re.sub(r"</v>", "", linea)
            linea = re.sub(r"^[A-Za-zÁÉÍÓÚÑáéíóúñ\s,]+:\s*", "", linea)

            if linea:
                buffer_texto.append(linea)

        if timestamp_actual and buffer_texto:
            conversaciones.append({
                "timestamp": timestamp_actual,
                "texto": " ".join(buffer_texto).strip()
            })

        return conversaciones

    # -----------------------------------------------------------------------
    # 2. EXTRACCIÓN DE ACTIVIDADES (SIN IA)
    # -----------------------------------------------------------------------
    def extraer_actividades(self, conversaciones):
        actividades = []
        actividad_actual = None

        patrones_inicio = [
            r"\binicio\b", r"\biniciar\b", r"\bcomienzo\b",
            r"\bempiezo\b", r"\bvoy a iniciar\b"
        ]
        patrones_fin = [
            r"\bfinalizo\b", r"\btermino\b",
            r"\bterminar\b", r"\bfinalizar\b"
        ]

        for conv in conversaciones:
            texto = conv["texto"].lower()
            timestamp = conv["timestamp"]

            if any(re.search(p, texto) for p in patrones_inicio):
                actividad_actual = {
                    "descripcion": conv["texto"],
                    "inicio": timestamp,
                    "fin": None
                }
            elif actividad_actual and any(re.search(p, texto) for p in patrones_fin):
                actividad_actual["fin"] = timestamp
                actividades.append(actividad_actual)
                actividad_actual = None

        return actividades

    # -----------------------------------------------------------------------
    # 3. CÁLCULO DE DURACIÓN
    # -----------------------------------------------------------------------
    def calcular_duracion_minutos(self, inicio, fin):
        h1, m1, s1 = map(int, inicio.split(":"))
        h2, m2, s2 = map(int, fin.split(":"))
        t1 = h1 * 3600 + m1 * 60 + s1
        t2 = h2 * 3600 + m2 * 60 + s2
        return max((t2 - t1) // 60, 0)

    # -----------------------------------------------------------------------
    # 4. PIPELINE DE PROCESAMIENTO
    # -----------------------------------------------------------------------
    def procesar_archivo(self, contenido):
        conversaciones = self.leer_archivo_vtt(contenido)
        actividades = self.extraer_actividades(conversaciones)

        resultado = []
        for a in actividades:
            if not a["fin"]:
                continue
            duracion = self.calcular_duracion_minutos(a["inicio"], a["fin"])
            if duracion <= 0:
                continue
            resultado.append({
                "actividad": a["descripcion"],
                "inicio": a["inicio"],
                "fin": a["fin"],
                "duracion_min": duracion
            })
        return resultado

    # -----------------------------------------------------------------------
    # 5. ENRIQUECIMIENTO CON IA (UNIDAD + PHVA)
    # -----------------------------------------------------------------------
    def _parse_json_seguro(self, texto):
        texto = texto.strip()
        if not texto:
            raise ValueError("❌ El LLM devolvió una respuesta vacía")
        if texto.startswith("```"):
            texto = re.sub(r"```json|```", "", texto).strip()
        inicio = min(
            [i for i in [texto.find("["), texto.find("{")] if i != -1],
            default=-1
        )
        if inicio > 0:
            texto = texto[inicio:]
        try:
            return json.loads(texto)
        except json.JSONDecodeError:
            print("❌ JSON inválido devuelto por el LLM")
            print("Respuesta cruda:")
            print(texto)
            raise

    def enriquecer_actividades(self, actividades, cargo, vicepresidencia):
        prompt = f"""
Eres un analista experto en Levantamiento de Cargas Laborales.

Contexto del colaborador:
- Cargo: {cargo}
- Vicepresidencia: {vicepresidencia}

Para cada actividad, debes:
- unidad_medida (ej: solicitudes, informes, reuniones, casos, desarrollos)
- phva (Planear, Hacer, Verificar, Actuar)

⚠️ REGLAS ESTRICTAS:
- Devuelve EXCLUSIVAMENTE un JSON válido
- NO incluyas texto antes o después
- NO expliques nada
- NO uses markdown
- Devuelve una LISTA del mismo tamaño que la entrada

Formato exacto de salida:
[
  {{
    "nombre": "texto corto",
    "unidad_medida": "texto",
    "phva": "Planear | Hacer | Verificar | Actuar"
  }}
]

Actividades de entrada:
{json.dumps(actividades, indent=2, ensure_ascii=False)}
"""
        reply = llm.invoke(prompt)
        return self._parse_json_seguro(reply.content)

    # -----------------------------------------------------------------------
    # 6. CONSTRUCCIÓN DE RESUMEN ESTRUCTURADO
    # -----------------------------------------------------------------------
    def construir_resumen_actividades(self, actividades_enriquecidas):
        resumen = ""
        for i, act in enumerate(actividades_enriquecidas, start=1):
            resumen += f"""
Actividad {i}:
- Descripción: {act['nombre']}
- Frecuencia: {act.get('frecuencia', 'No especificada')}
- Duración (minutos): {act.get('duracion_min')}
- Proceso del área: {act.get('proceso_area')}
- Volumen: {act.get('volumen', 'N/A')}
- Unidad de Medida: {act.get('unidad_medida')}
- Tipo de actividad (PHVA): {act.get('phva')}
- Autonomía: {act.get('autonomia', 'N/A')}%
"""
            if act.get("observaciones"):
                resumen += f"- Observaciones: {act['observaciones']}\n"
        return resumen

    # -----------------------------------------------------------------------
    # 7. RESUMEN PARA CONFIRMACIÓN DEL USUARIO
    # -----------------------------------------------------------------------
    def resumen_para_confirmacion(self, actividades_enriquecidas, contexto):
        resumen_actividades = self.construir_resumen_actividades(
            actividades_enriquecidas
        )
        prompt = f"""
Eres un asistente de Levantamiento de Cargas Laborales.

Contexto del colaborador:
- Cargo: {contexto['cargo']}
- Vicepresidencia: {contexto['vicepresidencia']}

A continuación se presenta un resumen de las actividades
identificadas a partir de reuniones y la información ingresada
por el colaborador.

{resumen_actividades}

Instrucciones:
- Resume la información de forma clara y ordenada
- No combines el Volumen con la Unidad de Medida,
  dalos por separado ya que el volumen es respecto a la Frecuencia
- Usa lenguaje sencillo y no técnico
- No agregues información nueva
- No hagas cálculos adicionales
- No corrijas ortografía a menos que sean la misma palabra en distinta capitalización
- Finaliza preguntando si la información es correcta o si desea hacer ajustes
"""
        reply = llm.invoke(prompt)
        return reply.content

    # -----------------------------------------------------------------------
    # 8. LECTURA DE ARCHIVO VTT DESDE FILEUPLOAD (JUPYTER)
    # -----------------------------------------------------------------------
    def obtener_contenido_vtt(self, upload_widget):
        if not upload_widget.value:
            raise ValueError("No se ha subido ningún archivo")
        valor = upload_widget.value
        archivo = valor[0] if isinstance(valor, tuple) else list(valor.values())[0]
        contenido_raw = archivo["content"]
        if isinstance(contenido_raw, memoryview):
            return contenido_raw.tobytes().decode("utf-8")
        elif isinstance(contenido_raw, bytes):
            return contenido_raw.decode("utf-8")
        else:
            raise TypeError("Tipo de contenido no soportado")

    # -----------------------------------------------------------------------
    # 8b. FUSIÓN DE ACTIVIDADES DIARIAS
    # -----------------------------------------------------------------------
    def fusionar_inputs_usuario(
        self, actividades_base, actividades_enriquecidas, inputs_usuario
    ):
        actividades_finales = []
        for base, ia, user in zip(
            actividades_base, actividades_enriquecidas, inputs_usuario
        ):
            actividades_finales.append({
                "nombre": ia["nombre"],
                "duracion_min": base["duracion_min"],
                "unidad_medida": ia["unidad_medida"],
                "phva": ia["phva"],
                "frecuencia": user["frecuencia"],
                "volumen": user["volumen"],
                "proceso_area": user["proceso_area"],
                "autonomia": user["autonomia"],
                "observaciones": user.get("observaciones", "")
            })
        return actividades_finales

    # -----------------------------------------------------------------------
    # 9. CONFIRMAR LA INFORMACIÓN POR PARTE DEL USUARIO
    # -----------------------------------------------------------------------
    def confirmar_informacion(self, resumen_texto):
        print("📋 RESUMEN PARA CONFIRMACIÓN\n")
        print(resumen_texto)
        respuesta = input(
            "\n¿La información es correcta? (si / no): "
        ).strip().lower()
        return respuesta == "si"

    # -----------------------------------------------------------------------
    # 10. CALCULAR MÉTRICAS
    # -----------------------------------------------------------------------
    def calcular_metricas(self, actividades):
        DIAS_LABORALES_MES = 21
        MINUTOS_JORNADA_MES = 220 * 60
        JORNADA_LABORAL_DIARIA = 8.5

        FACTOR_FRECUENCIA = {
            "Diario": 1,
            "Semanal": 1 / 5,
            "Quincenal": 1 / 10,
            "Mensual": 1 / 21,
            "Bimensual": 1 / 42,
            "Trimestral": 1 / 63,
            "Semestral": 1 / 126,
            "Anual": 1 / 252
        }

        carga_w_por_phva = {}
        cantidad_por_phva = {}
        carga_w_por_frecuencia = {}
        cantidad_por_frecuencia = {}
        carga_w_por_proceso_area = {}
        cantidad_por_proceso_area = {}

        total_carga_w_sin_tm = 0
        total_carga_trabajo_individual = 0
        total_minutos_diarios = 0

        for act in actividades:
            tiempo_base = act["duracion_min"]
            volumen = act["volumen"]
            frecuencia = act["frecuencia"]
            autonomia = act["autonomia"] / 100
            phva = act.get("phva", "SIN CLASIFICAR")
            proceso_area = act.get("proceso_area", "SIN CLASIFICAR")

            factor = FACTOR_FRECUENCIA.get(frecuencia, 0)
            minutos_diarios = tiempo_base * volumen * factor
            minutos_mes = minutos_diarios * DIAS_LABORALES_MES
            carga_w_sin_tm = minutos_mes / MINUTOS_JORNADA_MES
            carga_trabajo_individual = carga_w_sin_tm * autonomia

            act["metricas"] = {
                "minutos_diarios": round(minutos_diarios, 2),
                "minutos_mes": round(minutos_mes, 2),
                "carga_w_sin_tm": round(carga_w_sin_tm, 4),
                "carga_trabajo_individual": round(carga_trabajo_individual, 4)
            }

            total_carga_w_sin_tm += carga_w_sin_tm
            total_carga_trabajo_individual += carga_trabajo_individual
            total_minutos_diarios += minutos_diarios

            carga_w_por_phva[phva] = carga_w_por_phva.get(phva, 0) + carga_w_sin_tm
            cantidad_por_phva[phva] = cantidad_por_phva.get(phva, 0) + 1
            carga_w_por_frecuencia[frecuencia] = (
                carga_w_por_frecuencia.get(frecuencia, 0) + carga_w_sin_tm
            )
            cantidad_por_frecuencia[frecuencia] = (
                cantidad_por_frecuencia.get(frecuencia, 0) + 1
            )
            carga_w_por_proceso_area[proceso_area] = (
                carga_w_por_proceso_area.get(proceso_area, 0) + carga_w_sin_tm
            )
            cantidad_por_proceso_area[proceso_area] = (
                cantidad_por_proceso_area.get(proceso_area, 0) + 1
            )

        almuerzo = 1 / 8
        baño_agua_etc = (3 * 5) / 60
        break_15min = 15 / 60
        latencia_software = 10 / 60
        cliente_ext_o_int = 20 / 60
        tiempo_muerto = sum([
            almuerzo / JORNADA_LABORAL_DIARIA,
            baño_agua_etc / JORNADA_LABORAL_DIARIA,
            break_15min / JORNADA_LABORAL_DIARIA,
            latencia_software / JORNADA_LABORAL_DIARIA,
            cliente_ext_o_int / JORNADA_LABORAL_DIARIA
        ])
        factor_tiempo_neto = round(1 - tiempo_muerto, 4)

        def porcentajes(dic):
            total = sum(dic.values())
            return {
                k: round(v / total, 4) if total > 0 else 0
                for k, v in dic.items()
            }

        horas_diarias_requeridas = total_minutos_diarios / 60
        horas_netas_por_persona = JORNADA_LABORAL_DIARIA * factor_tiempo_neto

        return {
            "carga_trabajo_phva": {k: round(v, 4) for k, v in carga_w_por_phva.items()},
            "cantidad_actividades_phva": cantidad_por_phva,
            "porcentaje_actividades_phva": porcentajes(carga_w_por_phva),
            "carga_trabajo_frecuencia": {k: round(v, 4) for k, v in carga_w_por_frecuencia.items()},
            "cantidad_actividades_frecuencia": cantidad_por_frecuencia,
            "porcentaje_actividades_frecuencia": porcentajes(carga_w_por_frecuencia),
            "carga_trabajo_proceso_area": {
                k: round(v, 4) for k, v in carga_w_por_proceso_area.items()
            },
            "cantidad_actividades_proceso_area": cantidad_por_proceso_area,
            "porcentaje_actividades_proceso_area": porcentajes(carga_w_por_proceso_area),
            "total_carga_w_sin_tm": round(total_carga_w_sin_tm, 4),
            "total_carga_trabajo_individual": round(total_carga_trabajo_individual, 4),
            "minutos_diarios_empleados": round(total_minutos_diarios, 2),
            "horas_diarias_requeridas": round(horas_diarias_requeridas, 2),
            "jornada_laboral_diaria": JORNADA_LABORAL_DIARIA,
            "factor_tiempo_neto_productivo": factor_tiempo_neto,
            "horas_netas_efectivas_por_persona": round(horas_netas_por_persona, 2),
            "numero_personas_requeridas": round(
                (total_minutos_diarios * (1 + tiempo_muerto) / 60) / horas_netas_por_persona, 3
            ),
            "tiempo_muerto": round(tiempo_muerto, 4),
            "horas_diarias_requeridas_final": round(
                total_minutos_diarios * (1 + tiempo_muerto) / 60, 2
            )
        }

    # -----------------------------------------------------------------------
    # 11. ANÁLISIS IA PARA EL ANALISTA
    # -----------------------------------------------------------------------
    def analisis_analista_ia(self, actividades, metricas, contexto):
        prompt = f"""
Eres un ANALISTA SENIOR en Levantamiento de Cargas Laborales y Dimensionamiento Operativo.
Tu tarea NO es recalcular datos, sino INTERPRETAR, VALIDAR COHERENCIA y EMITIR JUICIO PROFESIONAL
a partir de la información suministrada.

=========================
CONTEXTO ORGANIZACIONAL
=========================
Cargo: {contexto.get("cargo")}
Vicepresidencia: {contexto.get("vicepresidencia")}

=========================
ACTIVIDADES ANALIZADAS
=========================
{json.dumps(actividades, indent=2, ensure_ascii=False)}

=========================
MÉTRICAS CALCULADAS
=========================
{json.dumps(metricas, indent=2, ensure_ascii=False)}

=========================
INSTRUCCIONES DE ANÁLISIS
=========================

1. Analiza el NIVEL DE CARGA LABORAL.
2. Evalúa la COHERENCIA del resultado.
3. Interpreta el BALANCE PHVA.
4. Identifica RIESGOS OPERATIVOS.
5. Detecta OPORTUNIDADES DE AUTOMATIZACIÓN O MEJORA.
6. Formula RECOMENDACIONES EJECUTIVAS.

Menciona los valores numéricos mientras das el análisis e interpreta su significado.

Responde en español con lenguaje profesional, estructurado en los bloques anteriores.
"""
        reply = llm.invoke(prompt)
        return reply.content

    # -----------------------------------------------------------------------
    # 12. CAPTURAR CARGO Y VICEPRESIDENCIA (WIDGETS)
    # -----------------------------------------------------------------------
    def capturar_contexto_usuario(self):
        import ipywidgets as widgets
        from IPython.display import display

        cargo = widgets.Text(
            description="Cargo:",
            placeholder="Ej: Analista de Datos",
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        vicepresidencia = widgets.Text(
            description="Vicepresidencia:",
            placeholder="Ej: Tecnología",
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        display(cargo, vicepresidencia)
        return {"cargo": cargo, "vicepresidencia": vicepresidencia}

    # -----------------------------------------------------------------------
    # 13. OBTENER VALORES DE CONTEXTO
    # -----------------------------------------------------------------------
    def obtener_contexto_valores(self, contexto_widgets):
        return {
            "cargo": contexto_widgets["cargo"].value,
            "vicepresidencia": contexto_widgets["vicepresidencia"].value
        }

    # -----------------------------------------------------------------------
    # 14–15. CAPTURA DE INPUTS POR ACTIVIDAD (WIDGETS — se conserva para
    #         el flujo de actividades detectadas en la grabación)
    # -----------------------------------------------------------------------
    def capturar_inputs_usuario(self):
        import ipywidgets as widgets
        from IPython.display import display

        frecuencia = widgets.Dropdown(
            options=self.FRECUENCIAS_VALIDAS,
            description="Frecuencia:",
            style={'description_width': '120px'}
        )
        volumen = widgets.IntText(
            description="Volumen:",
            value=1,
            style={'description_width': '120px'}
        )
        autonomia = widgets.IntSlider(
            description="Autonomía (%):",
            min=0, max=100, value=80,
            style={'description_width': '120px'}
        )
        proceso_area = widgets.Textarea(
            description="Proceso del área:",
            layout=widgets.Layout(width='500px', height='80px'),
            style={'description_width': '120px'}
        )
        observaciones = widgets.Textarea(
            description="Observaciones:",
            layout=widgets.Layout(width='500px', height='80px'),
            style={'description_width': '120px'}
        )
        display(frecuencia, volumen, autonomia, proceso_area, observaciones)
        return {
            "frecuencia": frecuencia,
            "volumen": volumen,
            "proceso_area": proceso_area,
            "autonomia": autonomia,
            "observaciones": observaciones
        }

    def obtener_actividades_unicas(self, actividades):
        vistas = set()
        unicas = []
        for act in actividades:
            nombre = act["actividad"].strip().lower()
            if nombre not in vistas:
                vistas.add(nombre)
                unicas.append(act)
        return unicas

    def capturar_inputs_usuario_actividades_unicas(self, actividades):
        actividades_unicas = self.obtener_actividades_unicas(actividades)
        inputs_por_actividad = {}
        for act in actividades_unicas:
            print(f"\n Actividad: {act['actividad']}")
            inputs_por_actividad[act["actividad"].strip().lower()] = (
                self.capturar_inputs_usuario()
            )
        return inputs_por_actividad

    def obtener_inputs_usuario_valores(self, inputs_widgets):
        return {
            "frecuencia": inputs_widgets["frecuencia"].value,
            "volumen": inputs_widgets["volumen"].value,
            "autonomia": inputs_widgets["autonomia"].value,
            "proceso_area": inputs_widgets["proceso_area"].value,
            "observaciones": inputs_widgets["observaciones"].value
        }

    # -----------------------------------------------------------------------
    # 16. EDITAR ACTIVIDADES (GENERAL)
    # -----------------------------------------------------------------------
    def editar_actividades(self, actividades):
        while True:
            if not actividades:
                print("\n⚠️ No hay actividades para editar.")
                break

            print("\n✏️ ACTIVIDADES DISPONIBLES:")
            for i, act in enumerate(actividades, start=1):
                print(f"{i}. {act['nombre']}")

            opcion = input(
                "\nIngrese el número de la actividad "
                "(o 'salir' para terminar ajustes): "
            ).strip().lower()

            if opcion == "salir":
                break

            if not opcion.isdigit() or not (1 <= int(opcion) <= len(actividades)):
                print("⚠️ Opción inválida")
                continue

            idx = int(opcion) - 1
            actividad = actividades[idx]

            print(f"\n🔧 Actividad seleccionada: {actividad['nombre']}")
            accion = input(
                "¿Qué deseas hacer? (editar / eliminar / cancelar): "
            ).strip().lower()

            if accion == "cancelar":
                continue

            if accion == "eliminar":
                actividad_original = copy.deepcopy(actividad)
                confirmacion = input(
                    f"⚠️ ¿Seguro que deseas eliminar '{actividad['nombre']}'? (si / no): "
                ).strip().lower()
                if confirmacion == "si":
                    actividades.pop(idx)
                    print("🗑️ Actividad eliminada.")
                    confirmar_final = input(
                        "¿Confirmas la eliminación? (si / no): "
                    ).strip().lower()
                    if confirmar_final != "si":
                        actividades.insert(idx, actividad_original)
                        print("↩️ Eliminación revertida.")
                continue

            if accion != "editar":
                print("⚠️ Acción no válida.")
                continue

            campos_editables = [
                "frecuencia", "volumen", "duracion_min",
                "autonomia", "observaciones", "unidad_medida", "phva"
            ]
            print("\nCampos editables:")
            for c in campos_editables:
                print(f"- {c} (actual: {actividad.get(c)})")

            campo = input("\nCampo a modificar: ").strip()
            if campo not in campos_editables:
                print("⚠️ Campo no editable")
                continue

            nuevo_valor = input("Nuevo valor: ").strip()
            if campo in ("volumen", "duracion_min", "autonomia"):
                nuevo_valor = int(nuevo_valor)

            actividad_original = copy.deepcopy(actividad)
            actividad[campo] = nuevo_valor

            print("\n🧾 RESUMEN DEL CAMBIO:")
            print(f"- Actividad: {actividad['nombre']}")
            print(f"- Campo modificado: {campo}")
            print(f"- Valor anterior: {actividad_original.get(campo)}")
            print(f"- Nuevo valor: {nuevo_valor}")

            confirmar_cambio = input(
                "\n¿Confirmas este cambio? (si / no): "
            ).strip().lower()
            if confirmar_cambio != "si":
                actividades[idx] = actividad_original
                print("↩️ Cambio revertido.")
            else:
                print("✅ Cambio confirmado.")

        return actividades

    # -----------------------------------------------------------------------
    # 17. FUSIÓN ACTIVIDADES NO DIARIAS
    # -----------------------------------------------------------------------
    def fusionar_inputs_usuario_nodiarias(
        self, actividades_base, actividades_enriquecidas
    ):
        actividades_finales_nodiarias = []
        for base, ia in zip(actividades_base, actividades_enriquecidas):
            actividades_finales_nodiarias.append({
                "nombre": base["nombre"],
                "duracion_min": base["duracion_min"],
                "unidad_medida": ia["unidad_medida"],
                "phva": ia["phva"],
                "frecuencia": base["frecuencia"],
                "volumen": base["volumen"],
                "proceso_area": base["proceso_area"],
                "autonomia": base["autonomia"],
                "observaciones": base.get("observaciones", "")
            })
        return actividades_finales_nodiarias

    # -----------------------------------------------------------------------
    # 18. VALIDACIÓN DE COHERENCIA GLOBAL
    # -----------------------------------------------------------------------
    def validar_coherencia_global(self, actividades, contexto):
        prompt = f"""
Actúa como un validador técnico de coherencia de actividades laborales.

El volumen representa el número de veces que se ejecuta la actividad
dentro del periodo definido por la frecuencia.

Tu tarea:
1. Detectar únicamente incoherencias internas.
2. No hacer juicios del rol ni usar información externa.

Debes validar únicamente:
- volumen × duración dentro del periodo de la frecuencia
- acumulaciones problemáticas dentro del mismo periodo
- duraciones extremadamente bajas o altas en relación con el volumen
- inconsistencias temporales si existen

⚠️ IMPORTANTE:
- Solo muestra las actividades donde detectes incoherencias.
- NO muestres actividades que estén correctas.

Formato de salida obligatorio:

------------------------------------------------------------
🔎 VALIDACIÓN DE COHERENCIA

Para cada actividad con incoherencia:

📌 Actividad: [Nombre exacto]

Campos evaluados:
- Frecuencia: [valor]
- Volumen: [valor]
- Duración por unidad: [valor + unidad]

Cálculo implícito:
[volumen × duración = total dentro del periodo]

Incoherencia detectada:
- [Descripción técnica breve]

------------------------------------------------------------

🧾 Resumen general:
- Total de actividades analizadas: [número]
- Actividades con incoherencias: [número]
- Errores matemáticos explícitos: [sí/no]

Al final devuelve SIEMPRE un bloque JSON así:

{{
    "actividades_incoherentes": ["Nombre 1", "Nombre 2"]
}}

Si no hay incoherencias:

{{
    "actividades_incoherentes": []
}}

=========================
CONTEXTO
=========================
Cargo: {contexto.get("cargo")}
Vicepresidencia: {contexto.get("vicepresidencia")}

=========================
ACTIVIDADES REGISTRADAS
=========================
{json.dumps(actividades, indent=2, ensure_ascii=False)}

Responde en español.
"""
        reply = llm.invoke(prompt)
        content = reply.content
        actividades_incoherentes = []
        try:
            json_start = content.rfind("{")
            if json_start != -1:
                json_text = content[json_start:]
                json_data = json.loads(json_text)
                actividades_incoherentes = json_data.get(
                    "actividades_incoherentes", []
                )
        except Exception:
            actividades_incoherentes = []
        return content, actividades_incoherentes

    # -----------------------------------------------------------------------
    # 18.1 EDITAR ACTIVIDADES INCOHERENTES
    # -----------------------------------------------------------------------
    def editar_actividades_incoherentes(self, actividades, actividades_incoherentes):
        actividades_filtradas = [
            act for act in actividades
            if act["nombre"] in actividades_incoherentes
        ]
        if not actividades_filtradas:
            print("\n✅ No hay actividades con incoherencias para editar.")
            return actividades

        while True:
            print("\n⚠️ ACTIVIDADES CON INCOHERENCIAS:")
            for i, act in enumerate(actividades_filtradas, start=1):
                print(f"{i}. {act['nombre']}")

            opcion = input(
                "\nSeleccione el número de la actividad "
                "(o 'salir' para terminar ajustes): "
            ).strip().lower()

            if opcion == "salir":
                break

            if not opcion.isdigit() or not (
                1 <= int(opcion) <= len(actividades_filtradas)
            ):
                print("⚠️ Opción inválida")
                continue

            actividad = actividades_filtradas[int(opcion) - 1]
            idx = next(
                i for i, a in enumerate(actividades)
                if a["nombre"] == actividad["nombre"]
            )

            print(f"\n🔧 Actividad seleccionada: {actividad['nombre']}")
            accion = input(
                "¿Qué deseas hacer? (editar / eliminar / cancelar): "
            ).strip().lower()

            if accion == "cancelar":
                continue

            if accion == "eliminar":
                confirmacion = input(
                    f"¿Confirmas eliminar '{actividad['nombre']}'? (si / no): "
                ).strip().lower()
                if confirmacion == "si":
                    actividades.pop(idx)
                    print("🗑️ Actividad eliminada.")
                continue

            if accion != "editar":
                print("⚠️ Acción no válida.")
                continue

            campos_editables = [
                "frecuencia", "volumen", "duracion_min",
                "autonomia", "observaciones", "unidad_medida", "phva"
            ]
            print("\nCampos editables:")
            for c in campos_editables:
                print(f"- {c} (actual: {actividad.get(c)})")

            campo = input("\nCampo a modificar: ").strip()
            if campo not in campos_editables:
                print("⚠️ Campo no editable")
                continue

            nuevo_valor = input("Nuevo valor: ").strip()
            if campo in ("volumen", "duracion_min", "autonomia"):
                nuevo_valor = int(nuevo_valor)

            actividad_original = copy.deepcopy(actividad)
            actividades[idx][campo] = nuevo_valor

            print("\n🧾 RESUMEN DEL CAMBIO:")
            print(f"- Actividad: {actividad['nombre']}")
            print(f"- Campo: {campo}")
            print(f"- Antes: {actividad_original.get(campo)}")
            print(f"- Después: {nuevo_valor}")

            confirmar = input("\n¿Confirmas el cambio? (si / no): ").strip().lower()
            if confirmar != "si":
                actividades[idx] = actividad_original
                print("↩️ Cambio revertido.")
            else:
                print("✅ Cambio confirmado.")

        return actividades

    # -----------------------------------------------------------------------
    # 19. RESUMEN AUTOMÁTICO DE CAMBIOS
    # -----------------------------------------------------------------------
    def generar_resumen_cambios(self, antes, despues):
        prompt = f"""
Eres un asistente que compara versiones de información.

REGLAS:
- No recalcules métricas.
- No hagas análisis.
- Solo describe qué cambió.
- Sé breve y claro.
- Si no hubo cambios, indícalo.

====================
ANTES
====================
{json.dumps(antes, indent=2, ensure_ascii=False)}

====================
DESPUÉS
====================
{json.dumps(despues, indent=2, ensure_ascii=False)}

Responde en español en formato claro y organizado.
"""
        reply = llm.invoke(prompt)
        return reply.content

    # =======================================================================
    # ★ NUEVO — FLUJO CONVERSACIONAL PARA ACTIVIDADES NO DIARIAS
    # =======================================================================

    # -----------------------------------------------------------------------
    # A. VALIDAR UNA ACTIVIDAD CON IA (coherencia individual)
    # -----------------------------------------------------------------------
    def _validar_coherencia_actividad_ia(self, actividad, min_disponibles=None):
        """
        Envía UNA actividad al LLM para que detecte incoherencias
        entre frecuencia, volumen y duración.

        Parámetros
        ----------
        actividad       : dict  — actividad a validar
        min_disponibles : float|None — minutos diarios restantes de la jornada
                          ya descontando las actividades previas confirmadas.
                          Si se proporciona, se usa como tope real en vez de 510.

        Devuelve:
            (hay_incoherencia: bool, explicacion: str, recomendaciones: list[str])
        """
        tope_diario = round(min_disponibles, 1) if min_disponibles is not None else 510
        tope_label  = (
            f"{tope_diario} min disponibles en la jornada "
            f"(descontando actividades ya registradas)"
            if min_disponibles is not None
            else "510 min (8.5 h, jornada completa)"
        )

        prompt = f"""
Actúa como validador técnico de coherencia de actividades laborales.

Tienes UNA actividad con los siguientes datos:
{json.dumps(actividad, indent=2, ensure_ascii=False)}

Definición clave:
- "volumen" = número de veces que se ejecuta la actividad dentro del
  periodo definido por "frecuencia".
- "duracion_min" = minutos que tarda UNA ejecución.

Tiempo disponible en la jornada diaria para validar: {tope_label}

Debes verificar:
1. ¿El volumen × duración_min supera el tiempo disponible en el periodo?
   Usa el tiempo disponible indicado arriba como tope para frecuencia Diaria.
   Para otras frecuencias multiplica por los días del periodo:
   - Semanal: {tope_diario} × 5
   - Quincenal: {tope_diario} × 10
   - Mensual: {tope_diario} × 21
   - Bimensual: {tope_diario} × 42
   - Trimestral: {tope_diario} × 63
   - Semestral: {tope_diario} × 126
   - Anual: {tope_diario} × 252
2. ¿La duración individual parece extremadamente corta (< 1 min) o
   larga (> 480 min) para el tipo de actividad?
3. ¿Hay alguna otra inconsistencia lógica evidente?

Si hay incoherencia, genera recomendaciones CONCRETAS y ACCIONABLES sobre
qué valores específicos cambiar (campo, valor actual → valor sugerido).
Por ejemplo: "Reduce el volumen de 10 a 2" o "Cambia la frecuencia de
Diario a Semanal" o "Disminuye la duración de 300 min a 60 min".

Responde EXCLUSIVAMENTE con un JSON válido, sin texto adicional:

{{
  "hay_incoherencia": true | false,
  "explicacion": "descripción breve y clara de la incoherencia, o cadena vacía si no hay",
  "recomendaciones": [
    "Recomendación concreta 1 (campo: valor actual → valor sugerido)",
    "Recomendación concreta 2 (si aplica)"
  ]
}}

Si no hay incoherencia, devuelve:
{{
  "hay_incoherencia": false,
  "explicacion": "",
  "recomendaciones": []
}}
"""
        reply = llm.invoke(prompt)
        try:
            resultado = self._parse_json_seguro(reply.content)
            return (
                resultado.get("hay_incoherencia", False),
                resultado.get("explicacion", ""),
                resultado.get("recomendaciones", []),
            )
        except Exception:
            return False, "", []

    # -----------------------------------------------------------------------
    # B. MOSTRAR INCOHERENCIA Y PERMITIR CORRECCIÓN INMEDIATA
    # -----------------------------------------------------------------------
    def _mostrar_incoherencia_y_corregir(
        self, actividad, explicacion, recomendaciones, min_disponibles=None
    ):
        """
        Informa al usuario de la incoherencia detectada, muestra
        recomendaciones concretas de corrección y permite ajustar el campo
        que desee antes de continuar.

        La re-validación usa los minutos disponibles reales (no 8.5 h fijas).

        Devuelve la actividad (corregida o sin cambios).
        """
        print("\n" + "─" * 60)
        print("⚠️  POSIBLE INCOHERENCIA DETECTADA")
        print("─" * 60)
        print(f"   {explicacion}")

        if recomendaciones:
            print("\n💡 Recomendaciones para corregirla:")
            for i, rec in enumerate(recomendaciones, start=1):
                print(f"   {i}. {rec}")
        print("─" * 60)

        while True:
            respuesta = input(
                "\n¿Deseas corregir algún dato? (si / no): "
            ).strip().lower()

            if respuesta != "si":
                print("✅ Se conserva la información tal como fue ingresada.")
                break

            campos = ["nombre", "frecuencia", "volumen", "duracion_min",
                      "autonomia", "proceso_area", "observaciones"]
            print("\nCampos disponibles:")
            for c in campos:
                print(f"  - {c}: {actividad.get(c, 'N/A')}")

            campo = input("\n¿Qué campo deseas corregir? ").strip()
            if campo not in campos:
                print("⚠️ Campo no reconocido. Intenta de nuevo.")
                continue

            nuevo_valor = input(f"Nuevo valor para '{campo}': ").strip()

            # Conversiones y validaciones por tipo de campo
            if campo in ("volumen", "duracion_min"):
                try:
                    nuevo_valor = int(nuevo_valor)
                except ValueError:
                    print("⚠️ Debe ser un número entero.")
                    continue
            if campo == "autonomia":
                try:
                    nuevo_valor = int(nuevo_valor)
                    if nuevo_valor not in range(5, 105, 5):
                        print("⚠️ Autonomía debe ser un múltiplo de 5 entre 5 y 100.")
                        continue
                except ValueError:
                    print("⚠️ Debe ser un número entero.")
                    continue
            if campo == "frecuencia" and nuevo_valor.capitalize() not in self.FRECUENCIAS_VALIDAS:
                print(
                    f"⚠️ Frecuencia no válida. Opciones: "
                    f"{', '.join(self.FRECUENCIAS_VALIDAS)}"
                )
                continue
            if campo == "frecuencia":
                nuevo_valor = nuevo_valor.capitalize()

            actividad[campo] = nuevo_valor
            print(f"✅ Campo '{campo}' actualizado a: {nuevo_valor}")

            # Re-validar tras la corrección, usando el mismo tope de minutos
            print("\n🔍 Re-validando coherencia con el nuevo valor...")
            hay_inc, expl, recs = self._validar_coherencia_actividad_ia(
                actividad, min_disponibles=min_disponibles
            )
            if hay_inc:
                print("\n⚠️  Aún se detecta una posible incoherencia:")
                print(f"   {expl}")
                if recs:
                    print("💡 Recomendaciones:")
                    for i, rec in enumerate(recs, start=1):
                        print(f"   {i}. {rec}")
            else:
                print("✅ Los datos son coherentes ahora.")
                break

        return actividad

    # -----------------------------------------------------------------------
    # C1. CALCULAR MÉTRICAS PARCIALES ACUMULADAS
    # -----------------------------------------------------------------------
    def _calcular_metricas_parciales(self, actividades_hasta_ahora):
        """
        Calcula métricas intermedias sobre la lista de actividades
        registradas hasta el momento (incluida la nueva que se está
        evaluando). Devuelve un dict con los valores internos que el LLM
        usará para el análisis, sin mostrarlos al usuario.
        """
        DIAS_LABORALES_MES   = 21
        MINUTOS_JORNADA_MES  = 220 * 60   # 13 200 min
        JORNADA_DIARIA       = 8.5        # horas

        FACTOR_FRECUENCIA = {
            "Diario":     1,
            "Semanal":    1 / 5,
            "Quincenal":  1 / 10,
            "Mensual":    1 / 21,
            "Bimensual":  1 / 42,
            "Trimestral": 1 / 63,
            "Semestral":  1 / 126,
            "Anual":      1 / 252,
        }

        # Tiempo muerto fijo (igual que en calcular_metricas)
        tiempo_muerto = sum([
            (1 / 8)        / JORNADA_DIARIA,   # almuerzo
            ((3 * 5) / 60) / JORNADA_DIARIA,   # baño/agua
            (15 / 60)      / JORNADA_DIARIA,   # break 15 min
            (10 / 60)      / JORNADA_DIARIA,   # latencia software
            (20 / 60)      / JORNADA_DIARIA,   # cliente ext/int
        ])
        factor_neto = 1 - tiempo_muerto
        horas_netas_por_persona = JORNADA_DIARIA * factor_neto

        total_min_diarios          = 0.0
        total_carga_w_sin_tm       = 0.0
        total_carga_individual     = 0.0
        metricas_por_actividad     = []

        for act in actividades_hasta_ahora:
            factor    = FACTOR_FRECUENCIA.get(act.get("frecuencia", ""), 0)
            min_dia   = act["duracion_min"] * act["volumen"] * factor
            min_mes   = min_dia * DIAS_LABORALES_MES
            carga_w   = min_mes / MINUTOS_JORNADA_MES
            carga_ind = carga_w * (act["autonomia"] / 100)

            total_min_diarios      += min_dia
            total_carga_w_sin_tm   += carga_w
            total_carga_individual += carga_ind

            metricas_por_actividad.append({
                "nombre":              act["nombre"],
                "frecuencia":          act.get("frecuencia"),
                "min_diarios":         round(min_dia,  2),
                "carga_w_sin_tm":      round(carga_w,  4),
                "carga_individual":    round(carga_ind, 4),
            })

        horas_diarias_requeridas = total_min_diarios / 60
        horas_con_tm = total_min_diarios * (1 + tiempo_muerto) / 60
        personas_req = horas_con_tm / horas_netas_por_persona if horas_netas_por_persona else 0

        return {
            # Internos — NUNCA mostrar al usuario directamente
            "total_min_diarios":          round(total_min_diarios, 2),
            "horas_diarias_requeridas":   round(horas_diarias_requeridas, 2),
            "horas_con_tiempo_muerto":    round(horas_con_tm, 2),
            "total_carga_w_sin_tm":       round(total_carga_w_sin_tm, 4),
            "total_carga_individual":     round(total_carga_individual, 4),
            "personas_requeridas":        round(personas_req, 3),
            "jornada_laboral_diaria":     JORNADA_DIARIA,
            "horas_netas_por_persona":    round(horas_netas_por_persona, 2),
            "factor_tiempo_neto":         round(factor_neto, 4),
            "num_actividades":            len(actividades_hasta_ahora),
            "metricas_por_actividad":     metricas_por_actividad,
        }

    # -----------------------------------------------------------------------
    # C2. ANÁLISIS CONTEXTUAL POR ACTIVIDAD (LLM)
    # -----------------------------------------------------------------------
    def _analisis_actividad_en_contexto(
        self, actividad_nueva, actividades_previas, metricas, contexto
    ):
        """
        Pide al LLM un análisis breve, contextual y en lenguaje natural
        sobre el impacto de la nueva actividad en la carga acumulada.

        El LLM recibe las métricas internas pero su respuesta NO debe
        exponer valores numéricos de carga ni dotación; debe traducirlos
        a observaciones cualitativas comprensibles para el colaborador.
        """
        prompt = f"""
Eres un analista experto en Levantamiento de Cargas Laborales.
Tu rol en este momento es acompañar al colaborador durante el registro
de sus actividades y darle retroalimentación útil tras registrar cada una.

=========================
CONTEXTO DEL COLABORADOR
=========================
Cargo           : {contexto.get("cargo")}
Vicepresidencia : {contexto.get("vicepresidencia")}

=========================
ACTIVIDAD RECIÉN REGISTRADA
=========================
{json.dumps(actividad_nueva, indent=2, ensure_ascii=False)}

=========================
ACTIVIDADES YA REGISTRADAS ANTES DE ESTA
=========================
{json.dumps(actividades_previas, indent=2, ensure_ascii=False) if actividades_previas else "Ninguna (esta es la primera actividad registrada)"}

=========================
MÉTRICAS INTERNAS CALCULADAS (para tu análisis — NO las expongas)
=========================
Estas métricas son acumuladas incluyendo la actividad recién registrada:
{json.dumps(metricas, indent=2, ensure_ascii=False)}

=========================
INSTRUCCIONES ESTRICTAS
=========================
1. Usa las métricas internas SOLO como base de tu interpretación.
   NO menciones ni repitas los valores numéricos de carga laboral,
   dotación, minutos diarios ni factores de tiempo al usuario.
   El usuario solo debe recibir observaciones en lenguaje natural.

2. Tu análisis debe cubrir EXACTAMENTE estos dos puntos,
   de forma breve (máximo 3-4 oraciones por punto):

   a) ¿Qué representa esta actividad para la carga del colaborador?
      (alta/media/baja demanda, frecuencia relevante, autonomía, etc.)

   b) ¿Cómo se ve la carga acumulada hasta ahora en relación con
      una jornada laboral normal? ¿Se nota algún patrón llamativo
      (concentración en cierto tipo de tareas, mucha dependencia
      de otros, actividades muy frecuentes o de larga duración)?

3. Si es la primera actividad, solo analiza el punto (a).

4. Cierra con una frase breve y motivadora que invite al colaborador
   a continuar con el registro.

5. NO hagas preguntas. NO solicites correcciones. NO emitas juicios
   sobre si los datos son correctos o no.

Responde en español, tono profesional pero cercano.
"""
        reply = llm.invoke(prompt)
        return reply.content

    # -----------------------------------------------------------------------
    # C. MOSTRAR RESUMEN DE UNA ACTIVIDAD Y PEDIR CONFIRMACIÓN
    #    (el análisis de carga se calcula internamente como contexto
    #     silencioso para la validación; NO se muestra al usuario)
    # -----------------------------------------------------------------------
    def _confirmar_actividad(self, actividad, numero, actividades_previas, contexto):
        """
        1. Calcula métricas parciales acumuladas (uso interno / analista).
        2. Muestra el resumen de la actividad al usuario.
        3. Pide confirmación.

        El análisis de carga queda guardado en actividad['_analisis_interno']
        para uso posterior del analista; el colaborador no lo ve.

        Devuelve True si el usuario confirma, False si quiere rehacerla.
        """
        # --- Métricas parciales (silenciosas) ---
        todas    = actividades_previas + [actividad]
        metricas = self._calcular_metricas_parciales(todas)

        # Guardar análisis interno sin mostrarlo al usuario
        analisis_interno = self._analisis_actividad_en_contexto(
            actividad_nueva     = actividad,
            actividades_previas = actividades_previas,
            metricas            = metricas,
            contexto            = contexto,
        )
        actividad["_analisis_interno"] = analisis_interno

        # --- Resumen visible para el colaborador ---
        print("\n" + "═" * 60)
        print(f"  RESUMEN — Actividad {numero}")
        print("═" * 60)
        print(f"  Nombre            : {actividad.get('nombre')}")
        if actividad.get("descripcion"):
            print(f"  Descripción       : {actividad.get('descripcion')}")
        print(f"  Frecuencia        : {actividad.get('frecuencia')}")
        print(f"  Volumen           : {actividad.get('volumen')} "
              f"vez/veces por periodo")
        print(f"  Duración          : {actividad.get('duracion_min')} min "
              f"por ejecución")
        print(f"  Personas          : {actividad.get('personas')} "
              f"(autonomía: {actividad.get('autonomia')}%)")
        print(f"  Proceso del área  : {actividad.get('proceso_area') or '—'}")
        if actividad.get("observaciones"):
            print(f"  Observaciones     : {actividad.get('observaciones')}")
        print("═" * 60)

        resp = input(
            "\n¿Esta información es correcta? (si / no): "
        ).strip().lower()
        return resp == "si"

    # -----------------------------------------------------------------------
    # D. RECOGER UNA ACTIVIDAD COMPLETA DE FORMA CONVERSACIONAL
    # -----------------------------------------------------------------------
    def _preguntar_actividad(self, numero):
        """
        Conduce el diálogo para capturar todos los campos de
        una actividad y devuelve el diccionario con los datos.

        Autonomía: se elige entre opciones de 5 en 5 (5% → 100%).
        """
        print("\n" + "─" * 60)
        print(f"  📋  ACTIVIDAD {numero}")
        print("─" * 60)

        # --- Nombre / descripción ---
        nombre = ""
        while not nombre:
            nombre = input(
                f"\n¿Cuál es la actividad {numero}? "
                "(describe brevemente qué haces): "
            ).strip()
            if not nombre:
                print("⚠️ Por favor, ingresa una descripción.")

        descripcion = input(
            "Si quieres, amplía la descripción "
            "(o presiona Enter para continuar): "
        ).strip()

        # --- Frecuencia ---
        frecuencia = None
        opciones_str = "  ".join(
            f"{i+1}) {f}" for i, f in enumerate(self.FRECUENCIAS_VALIDAS)
        )
        print(f"\n¿Con qué frecuencia realizas esta actividad?")
        print(f"  {opciones_str}")
        while frecuencia is None:
            resp = input("Selecciona el número o escribe la frecuencia: ").strip()
            if resp.isdigit() and 1 <= int(resp) <= len(self.FRECUENCIAS_VALIDAS):
                frecuencia = self.FRECUENCIAS_VALIDAS[int(resp) - 1]
            elif resp.capitalize() in self.FRECUENCIAS_VALIDAS:
                frecuencia = resp.capitalize()
            else:
                print(
                    "⚠️ Opción no válida. "
                    f"Elige un número del 1 al {len(self.FRECUENCIAS_VALIDAS)}."
                )

        # --- Volumen ---
        volumen = None
        print(
            f"\n¿Cuántas veces realizas esta actividad por periodo "
            f"({frecuencia.lower()})?"
        )
        print("  (Ej: si es Diario y la haces 3 veces al día → escribe 3)")
        while volumen is None:
            resp = input("Volumen: ").strip()
            if resp.isdigit() and int(resp) > 0:
                volumen = int(resp)
            else:
                print("⚠️ Ingresa un número entero mayor a 0.")

        # --- Duración ---
        duracion_min = None
        print(
            "\n¿Cuántos minutos te toma completar UNA ejecución "
            "de esta actividad?"
        )
        while duracion_min is None:
            resp = input("Duración (minutos): ").strip()
            if resp.isdigit() and int(resp) > 0:
                duracion_min = int(resp)
            else:
                print("⚠️ Ingresa un número entero mayor a 0.")

        # --- Autonomía (5% a 100% de 5 en 5) ---
        autonomia_opciones = list(range(5, 105, 5))   # [5, 10, 15, ..., 100]
        autonomia = None
        print(
            "\n¿Qué porcentaje de esta actividad recae sobre ti?"
        )
        print(
            "  Opciones disponibles (de 5 en 5):"
        )
        # Mostrar en filas de 10 para no saturar la pantalla
        fila = "  " + "  ".join(f"{p}%" for p in autonomia_opciones)
        print(fila)
        print(
            "  (Ej: si eres el único responsable → 100%  |  "
            "si compartes al 50% con otro → 50%)"
        )
        while autonomia is None:
            resp = input("Autonomía (%): ").strip().replace("%", "").strip()
            if resp.isdigit() and int(resp) in autonomia_opciones:
                autonomia = int(resp)
            else:
                print(
                    f"⚠️ Elige un valor de 5 en 5 entre 5% y 100%. "
                    f"Ej: 5, 10, 15, ... , 95, 100."
                )

        # --- Proceso del área ---
        proceso_area = input(
            "\n¿A qué proceso del área pertenece esta actividad? "
            "(Ej: Gestión de proveedores, Reportes, Atención al cliente): "
        ).strip()

        # --- Observaciones ---
        observaciones = input(
            "\n¿Tienes alguna observación adicional sobre esta actividad? "
            "(presiona Enter si no): "
        ).strip()

        return {
            "nombre":        nombre,
            "descripcion":   descripcion,
            "frecuencia":    frecuencia,
            "volumen":       volumen,
            "duracion_min":  duracion_min,
            "autonomia":     autonomia,   # % elegido por el usuario (múltiplo de 5)
            "proceso_area":  proceso_area,
            "observaciones": observaciones,
        }

    # -----------------------------------------------------------------------
    # E.0  HELPERS RÁPIDOS (sin LLM)
    # -----------------------------------------------------------------------
    def _calcular_minutos_diarios_acumulados(self, actividades):
        """Minutos diarios equivalentes acumulados (factor de frecuencia)."""
        FACTOR_FRECUENCIA = {
            "Diario": 1, "Semanal": 1/5, "Quincenal": 1/10,
            "Mensual": 1/21, "Bimensual": 1/42, "Trimestral": 1/63,
            "Semestral": 1/126, "Anual": 1/252,
        }
        total = 0.0
        for act in actividades:
            factor = FACTOR_FRECUENCIA.get(act.get("frecuencia", ""), 0)
            total += act["duracion_min"] * act["volumen"] * factor
        return round(total, 2)

    def _es_duplicada(self, nombre_nuevo, actividades_existentes, umbral=0.75):
        """
        Detecta si el nombre de la nueva actividad es muy similar al de
        alguna actividad ya registrada, usando similitud de caracteres
        (SequenceMatcher) como heurística rápida sin LLM.

        Retorna (bool, str|None) — (es_duplicada, nombre_original_similar)
        """
        from difflib import SequenceMatcher
        nombre_norm = nombre_nuevo.strip().lower()
        for act in actividades_existentes:
            existente_norm = act["nombre"].strip().lower()
            ratio = SequenceMatcher(None, nombre_norm, existente_norm).ratio()
            if ratio >= umbral:
                return True, act["nombre"]
        return False, None

    # -----------------------------------------------------------------------
    # E. FLUJO PRINCIPAL CONVERSACIONAL
    # -----------------------------------------------------------------------
    def capturar_actividades_conversacional(self, contexto):
        """
        Conduce al usuario a través de un diálogo para registrar todas
        sus actividades.

        Por cada actividad:
          1. Recoge los datos conversacionalmente.
          2. Detecta si ya fue registrada anteriormente (duplicado).
          3. Valida coherencia con el LLM usando los minutos restantes reales.
          4. Muestra recomendaciones concretas si hay incoherencia.
          5. Muestra "✅ Coherencia OK" si no hay problema.
          6. Muestra resumen y pide confirmación.
          7. Tras confirmar, actualiza la barra de carga y emite alerta
             basada en minutos RESTANTES (no en 8.5 h fijas).
        """
        JORNADA_MIN = 8.5 * 60   # 510 minutos

        print("\n" + "═" * 60)
        print("  🗂️  REGISTRO DE ACTIVIDADES — MODO CONVERSACIONAL")
        print("═" * 60)
        print(
            "\nA continuación te haré preguntas sobre cada actividad que realizas.\n"
            "Puedes registrar todas las que quieras.\n"
            "Cuando termines, escribe 'no' cuando te pregunte si hay otra actividad."
        )

        actividades = []
        numero = 1
        tope_alcanzado = False

        while True:
            # ── Capturar + validar + confirmar ────────────────────────────
            while True:
                actividad = self._preguntar_actividad(numero)

                # ── 1. Detección de duplicados ────────────────────────────
                es_dup, nombre_similar = self._es_duplicada(
                    actividad["nombre"], actividades
                )
                if es_dup:
                    print("\n" + "─" * 60)
                    print("⚠️  POSIBLE ACTIVIDAD DUPLICADA")
                    print("─" * 60)
                    print(
                        f"   La actividad \"{actividad['nombre']}\" es muy similar\n"
                        f"   a \"{nombre_similar}\", que ya fue registrada."
                    )
                    decision = input(
                        "\n   ¿Es realmente una actividad diferente? (si / no): "
                    ).strip().lower()
                    if decision != "si":
                        print("🔄 Descartada. Ingresa una actividad distinta.\n")
                        continue   # vuelve a pedir la actividad

                # ── 2. Calcular minutos disponibles restantes ─────────────
                min_acum_previo = self._calcular_minutos_diarios_acumulados(actividades)
                min_disponibles = max(JORNADA_MIN - min_acum_previo, 0)

                # ── 3. Validar coherencia con tope real de minutos ─────────
                print("\n🔍 Revisando coherencia de los datos ingresados...")
                hay_incoherencia, explicacion, recomendaciones = (
                    self._validar_coherencia_actividad_ia(
                        actividad,
                        min_disponibles=min_disponibles if min_acum_previo > 0 else None,
                    )
                )

                if hay_incoherencia:
                    actividad = self._mostrar_incoherencia_y_corregir(
                        actividad,
                        explicacion,
                        recomendaciones,
                        min_disponibles=min_disponibles if min_acum_previo > 0 else None,
                    )
                else:
                    print("✅ Coherencia OK — los datos ingresados son consistentes.")

                # ── 4. Confirmar actividad ────────────────────────────────
                if self._confirmar_actividad(
                    actividad           = actividad,
                    numero              = numero,
                    actividades_previas = actividades,
                    contexto            = contexto,
                ):
                    actividades.append(actividad)
                    print(f"\n✅ Actividad {numero} registrada.")
                    break
                else:
                    print(
                        f"\n🔄 Volvemos a registrar la actividad {numero}. "
                        "Ingresa los datos de nuevo.\n"
                    )

            # ── Barra de carga y alerta por minutos restantes ─────────────
            min_acum  = self._calcular_minutos_diarios_acumulados(actividades)
            horas_acum = min_acum / 60
            min_rest  = max(JORNADA_MIN - min_acum, 0)
            horas_rest = min_rest / 60

            pct    = min(min_acum / JORNADA_MIN, 1.0)
            bloques = int(pct * 20)
            barra  = "█" * bloques + "░" * (20 - bloques)
            print(
                f"\n  ⏱️  Carga acumulada: [{barra}] "
                f"{horas_acum:.1f} h de 8.5 h hábiles  "
                f"({horas_rest:.1f} h disponibles)"
            )

            if min_acum >= JORNADA_MIN and not tope_alcanzado:
                tope_alcanzado = True
                print("\n" + "⚠️ " * 20)
                print(
                    "  ATENCIÓN: Las actividades registradas ya ocupan\n"
                    "  TODA la jornada laboral disponible (8.5 horas).\n"
                    "  Si agregas más, la carga superará la capacidad diaria."
                )
                print("⚠️ " * 20)
                continuar = input(
                    "\n  ¿Deseas seguir registrando actividades de todas formas? "
                    "(si / no): "
                ).strip().lower()
                if continuar != "si":
                    break

            elif min_acum >= JORNADA_MIN and tope_alcanzado:
                exceso_min = round(min_acum - JORNADA_MIN, 1)
                print(
                    f"  ⚠️  Ya superaste la jornada diaria en "
                    f"~{exceso_min} min equivalentes."
                )

            # ── ¿Hay más actividades? ─────────────────────────────────────
            print("\n" + "─" * 60)
            otra = input(
                "¿Tienes otra actividad para registrar? (si / no): "
            ).strip().lower()

            if otra != "si":
                break

            numero += 1

        print("\n" + "═" * 60)
        print(f"  ✅  Se registraron {len(actividades)} actividad(es) en total.")
        print("═" * 60)
        return actividades

    # -----------------------------------------------------------------------
    # E.1  CAPTURAR CONTEXTO DEL COLABORADOR (conversacional, sin widgets)
    # -----------------------------------------------------------------------
    def capturar_contexto_conversacional(self):
        """
        Solicita cargo y vicepresidencia de forma conversacional al inicio
        del pipeline, con confirmación para evitar errores de tipeo.

        Retorna
        -------
        dict  {"cargo": str, "vicepresidencia": str}
        """
        print("\n" + "═" * 60)
        print("  👤  INFORMACIÓN DEL COLABORADOR")
        print("═" * 60)

        while True:
            # --- Cargo ---
            cargo = ""
            while not cargo:
                cargo = input(
                    "\n¿Cuál es tu cargo actual? "
                    "(Ej: Analista de Operaciones, Coordinador de Proyectos): "
                ).strip()
                if not cargo:
                    print("⚠️ Por favor, ingresa tu cargo.")

            # --- Vicepresidencia ---
            vicepresidencia = ""
            while not vicepresidencia:
                vicepresidencia = input(
                    "\n¿A qué vicepresidencia o área perteneces? "
                    "(Ej: Tecnología, Operaciones, Finanzas): "
                ).strip()
                if not vicepresidencia:
                    print("⚠️ Por favor, ingresa la vicepresidencia.")

            # --- Confirmación ---
            print("\n" + "─" * 60)
            print("  Verifica que la información esté correcta:")
            print(f"    Cargo           : {cargo}")
            print(f"    Vicepresidencia : {vicepresidencia}")
            print("─" * 60)

            confirma = input(
                "\n¿La información es correcta? (si / no): "
            ).strip().lower()

            if confirma == "si":
                print(f"\n✅ Contexto registrado: {cargo} — {vicepresidencia}")
                return {"cargo": cargo, "vicepresidencia": vicepresidencia}

            print("\n🔄 Volvamos a ingresar la información.\n")

    # -----------------------------------------------------------------------
    # G. GENERAR EXCEL DESCARGABLE CON LOS DATOS DEL COLABORADOR
    # -----------------------------------------------------------------------
    def generar_excel_actividades(
        self,
        actividades,
        contexto,
        ruta_plantilla=None,
        ruta_salida="levantamiento_cargas.xlsx"
    ):
        """
        Rellena el formato estándar FORMATO_ANALISIS_DE_CARGA.xlsx con las
        actividades registradas por el colaborador y lo guarda como nuevo archivo.

        Estructura esperada de la plantilla (hoja "FORMATO"):
          Fila 1  → encabezados (A:K)
          Fila 2+ → filas de datos pre-formateadas con bordes

        Columnas del formato:
          A  NO.
          B  DESCRIPCIÓN DE LA ACTIVIDAD
          C  UNIDAD DE MEDIDA
          D  FRECUENCIA
          E  VOL.(SEGÚN FRECUENCIA)
          F  TIEMPO ESTIMADO POR UNIDAD EN MINUTOS
          G  PROCESO DEL ÁREA
          H  AUTONOMÍA DE LA TAREA (0-100%)
          I  TIPO DE ACTIVIDAD (DENTRO DEL PHVA)
          J  VICEPRESIDENCIA
          K  OBSERVACIONES

        Si no se proporciona plantilla, genera un archivo nuevo con el mismo
        esquema de columnas y formato básico.

        Parámetros
        ----------
        actividades    : list[dict]  — actividades enriquecidas y confirmadas
        contexto       : dict        — {"cargo": ..., "vicepresidencia": ...}
        ruta_plantilla : str|None    — ruta a FORMATO_ANALISIS_DE_CARGA.xlsx
        ruta_salida    : str         — nombre del archivo de salida

        Retorna
        -------
        str  — ruta del archivo generado
        """
        from openpyxl import Workbook, load_workbook
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        from openpyxl.utils import get_column_letter
        import copy

        vicepresidencia = contexto.get("vicepresidencia", "")

        if ruta_plantilla:
            # ── MODO PLANTILLA ────────────────────────────────────────────────
            wb = load_workbook(ruta_plantilla)

            # Usar hoja FORMATO si existe, si no la activa
            ws = wb["FORMATO"] if "FORMATO" in wb.sheetnames else wb.active

            # Detectar fila de encabezados buscando "DESCRIPCIÓN DE LA ACTIVIDAD"
            fila_encabezado = None
            for fila in ws.iter_rows():
                for celda in fila:
                    if (
                        celda.value
                        and "DESCRIPCI" in str(celda.value).upper()
                        and "ACTIVIDAD" in str(celda.value).upper()
                    ):
                        fila_encabezado = celda.row
                        break
                if fila_encabezado:
                    break

            if fila_encabezado is None:
                raise ValueError(
                    "No se encontró la fila de encabezados en la plantilla."
                )

            # Construir mapa: NOMBRE_ENCABEZADO_NORMALIZADO → col_index
            col_map = {}
            for celda in ws[fila_encabezado]:
                if celda.value:
                    key = str(celda.value).strip().upper()
                    col_map[key] = celda.column

            fila_inicio = fila_encabezado + 1

            # Limpiar celdas residuales en el rango de datos
            # (ej: A30=31 que existe en el template original)
            for r in range(fila_inicio, ws.max_row + 1):
                for c in range(1, ws.max_column + 1):
                    ws.cell(row=r, column=c).value = None

            # Escribir cada actividad en su fila correspondiente
            for i, act in enumerate(actividades):
                fila = fila_inicio + i
                datos = _actividad_a_fila(act, numero=i + 1,
                                          vicepresidencia=vicepresidencia)

                for nombre_col_normalizado, valor in datos.items():
                    col_idx = col_map.get(nombre_col_normalizado)
                    if col_idx:
                        celda = ws.cell(row=fila, column=col_idx, value=valor)
                        # Aplicar formato de porcentaje a columna Autonomía
                        if "AUTONOM" in nombre_col_normalizado:
                            celda.number_format = "0%"
                        # Centrar columnas cortas; wrap en columnas largas
                        if nombre_col_normalizado in (
                            "NO.", "FRECUENCIA", "VOL.(SEGÚN FRECUENCIA)",
                            "TIEMPO ESTIMADO POR UNIDAD EN MINUTOS",
                            "AUTONOMÍA DE LA TAREA (0-100%)",
                            "TIPO DE ACTIVIDAD (DENTRO DEL PHVA)",
                        ):
                            celda.alignment = Alignment(
                                horizontal="center", vertical="center"
                            )
                        else:
                            celda.alignment = Alignment(
                                vertical="center", wrap_text=True
                            )

        else:
            # ── MODO SIN PLANTILLA: crear desde cero con mismo esquema ────────
            ENCABEZADOS = [
                "NO.",
                "DESCRIPCIÓN DE LA ACTIVIDAD",
                "UNIDAD DE MEDIDA",
                "FRECUENCIA",
                "VOL.(SEGÚN FRECUENCIA)",
                "TIEMPO ESTIMADO POR UNIDAD EN MINUTOS",
                "PROCESO DEL ÁREA",
                "AUTONOMÍA DE LA TAREA (0-100%)",
                "TIPO DE ACTIVIDAD (DENTRO DEL PHVA)",
                "VICEPRESIDENCIA",
                "OBSERVACIONES",
            ]
            ANCHOS = [7, 55, 22, 18, 22, 22, 22, 18, 18, 20, 25]

            borde = Border(
                top    = Side(style="thin"),
                bottom = Side(style="thin"),
                left   = Side(style="thin"),
                right  = Side(style="thin"),
            )

            wb = Workbook()
            ws = wb.active
            ws.title = "FORMATO"

            # Encabezados
            ws.append(ENCABEZADOS)
            for col_idx, _ in enumerate(ENCABEZADOS, start=1):
                c = ws.cell(row=1, column=col_idx)
                c.font      = Font(bold=True, size=14)
                c.alignment = Alignment(
                    horizontal="center", vertical="center", wrap_text=True
                )
                c.border    = borde
            ws.row_dimensions[1].height = 59

            for col_idx, ancho in enumerate(ANCHOS, start=1):
                ws.column_dimensions[get_column_letter(col_idx)].width = ancho

            # Filas de datos
            for i, act in enumerate(actividades):
                datos = _actividad_a_fila(act, numero=i + 1,
                                          vicepresidencia=vicepresidencia)
                fila_vals = [datos.get(enc.upper(), "") for enc in ENCABEZADOS]
                ws.append(fila_vals)
                fila_actual = ws.max_row
                for col_idx in range(1, len(ENCABEZADOS) + 1):
                    c = ws.cell(row=fila_actual, column=col_idx)
                    c.border    = borde
                    c.alignment = Alignment(vertical="center", wrap_text=True)
                    if col_idx == 8:   # Autonomía
                        c.number_format = "0%"

        wb.save(ruta_salida)
        print(f"\n📥 Excel generado: {ruta_salida}")
        return ruta_salida

    # -----------------------------------------------------------------------
    # H. GUARDAR EN BIGQUERY
    # -----------------------------------------------------------------------
    def guardar_en_bigquery(
        self,
        actividades,
        contexto,
        project_id,
        dataset_id,
        table_id,
        credentials_path=None
    ):
        """
        Inserta las actividades finales en una tabla de BigQuery.

        La tabla se crea automáticamente si no existe (schema inferido).
        Si ya existe, se hace append de las filas nuevas.

        Parámetros
        ----------
        actividades      : list[dict]  — actividades enriquecidas
        contexto         : dict        — {"cargo": ..., "vicepresidencia": ...}
        project_id       : str         — GCP project ID
        dataset_id       : str         — dataset de BigQuery
        table_id         : str         — tabla de destino
        credentials_path : str|None    — ruta a service account JSON;
                                         None usa Application Default Credentials
        """
        from google.cloud import bigquery
        from google.oauth2 import service_account
        from datetime import datetime, timezone

        # Credenciales
        if credentials_path:
            creds  = service_account.Credentials.from_service_account_file(
                credentials_path,
                scopes=["https://www.googleapis.com/auth/cloud-platform"]
            )
            client = bigquery.Client(project=project_id, credentials=creds)
        else:
            # Application Default Credentials
            client = bigquery.Client(project=project_id)

        tabla_ref   = f"{project_id}.{dataset_id}.{table_id}"
        timestamp   = datetime.now(timezone.utc).isoformat()

        filas = []
        for act in actividades:
            filas.append({
                "timestamp_carga":      timestamp,
                "cargo":                contexto.get("cargo", ""),
                "vicepresidencia":      contexto.get("vicepresidencia", ""),
                "descripcion_actividad":act.get("nombre", ""),
                "frecuencia":           act.get("frecuencia", ""),
                "volumen":              act.get("volumen"),
                "duracion_min":         act.get("duracion_min"),
                "personas":             act.get("personas"),
                "autonomia_pct":        act.get("autonomia"),
                "proceso_area":         act.get("proceso_area", ""),
                "unidad_medida":        act.get("unidad_medida", ""),
                "phva":                 act.get("phva", ""),
                "observaciones":        act.get("observaciones", ""),
            })

        errores = client.insert_rows_json(tabla_ref, filas)

        if not errores:
            print(
                f"\n✅ {len(filas)} fila(s) insertada(s) en "
                f"{tabla_ref}"
            )
        else:
            print(f"\n⚠️ Errores al insertar en BigQuery:")
            for err in errores:
                print(f"   {err}")

    # -----------------------------------------------------------------------
    # F. PIPELINE COMPLETO PARA ACTIVIDADES CONVERSACIONALES
    # -----------------------------------------------------------------------
    def procesar_actividades_conversacional(
        self,
        ruta_plantilla_excel=None,
        ruta_salida_excel="levantamiento_cargas.xlsx",
        bq_project_id=None,
        bq_dataset_id=None,
        bq_table_id="levantamiento_cargas",
        bq_credentials_path=None,
    ):
        """
        Pipeline de extremo a extremo para el flujo conversacional.

        Flujo:
        1. Solicita cargo y vicepresidencia (conversacional).
        2. Captura actividades con validación de coherencia y análisis
           interno de carga por actividad (invisible para el colaborador).
        3. Enriquece con IA (unidad_medida + phva).
        4. Fusiona datos base + IA.
        5. Muestra resumen global y pide confirmación final.
        6. Genera Excel descargable.
        7. Inserta en BigQuery (si se configuran los parámetros BQ).

        Parámetros
        ----------
        ruta_plantilla_excel : str|None — plantilla Excel existente (opcional)
        ruta_salida_excel    : str      — nombre del Excel de salida
        bq_project_id        : str|None — GCP project (None = omite BQ)
        bq_dataset_id        : str|None — dataset de BigQuery
        bq_table_id          : str      — tabla de destino en BQ
        bq_credentials_path  : str|None — JSON de credenciales (None = ADC)

        Retorna
        -------
        list[dict]  — actividades enriquecidas y confirmadas
        """
        # PASO 1 — Capturar contexto del colaborador
        contexto = self.capturar_contexto_conversacional()

        # PASO 2 — Captura conversacional con análisis interno por actividad
        actividades_base = self.capturar_actividades_conversacional(contexto)

        if not actividades_base:
            print("\n⚠️ No se registraron actividades.")
            return []

        # PASO 3 — Enriquecimiento IA
        print("\n🤖 Enriqueciendo actividades con IA...")
        actividades_ia = self.enriquecer_actividades(
            actividades_base,
            contexto["cargo"],
            contexto["vicepresidencia"],
        )

        # PASO 4 — Fusión
        actividades_finales = self.fusionar_inputs_usuario_nodiarias(
            actividades_base, actividades_ia
        )

        # PASO 5 — Resumen global y confirmación final
        #          Loop: si el usuario quiere corregir → editor → nuevo resumen
        while True:
            print("\n📋 Generando resumen para tu confirmación final...")
            resumen = self.resumen_para_confirmacion(actividades_finales, contexto)
            confirmado = self.confirmar_informacion(resumen)

            if confirmado:
                break

            # Usuario quiere corregir algo
            print("\n✏️ Abriendo editor de actividades...")
            actividades_finales = self.editar_actividades(actividades_finales)

            # Re-enriquecer con IA las actividades que cambiaron
            # (solo si hubo cambios en campos que el LLM maneja)
            print(
                "\n🤖 Re-analizando actividades con la información actualizada..."
            )
            actividades_ia_nuevo = self.enriquecer_actividades(
                actividades_finales,
                contexto["cargo"],
                contexto["vicepresidencia"],
            )
            # Actualizar unidad_medida y phva desde el nuevo enriquecimiento
            for act, ia in zip(actividades_finales, actividades_ia_nuevo):
                act["unidad_medida"] = ia.get("unidad_medida", act.get("unidad_medida", ""))
                act["phva"]          = ia.get("phva",          act.get("phva", ""))

            print("\n✅ Análisis actualizado. Generando nuevo resumen...")

        # PASO 6 — Generar Excel
        self.generar_excel_actividades(
            actividades   = actividades_finales,
            contexto      = contexto,
            ruta_plantilla= ruta_plantilla_excel,
            ruta_salida   = ruta_salida_excel,
        )

        # PASO 7 — BigQuery (solo si se configuraron parámetros)
        if bq_project_id and bq_dataset_id:
            self.guardar_en_bigquery(
                actividades      = actividades_finales,
                contexto         = contexto,
                project_id       = bq_project_id,
                dataset_id       = bq_dataset_id,
                table_id         = bq_table_id,
                credentials_path = bq_credentials_path,
            )

        return actividades_finales


# ---------------------------------------------------------------------------
# Función auxiliar (module-level) para mapear un dict de actividad
# a los nombres de columna del Excel estándar
# ---------------------------------------------------------------------------
def _actividad_a_fila(act, numero=None, vicepresidencia=""):
    """
    Convierte un dict de actividad a las 11 columnas del formato estándar
    FORMATO_ANALISIS_DE_CARGA.xlsx (hoja FORMATO).

    Columnas (en orden del template):
      A  NO.
      B  DESCRIPCIÓN DE LA ACTIVIDAD
      C  UNIDAD DE MEDIDA
      D  FRECUENCIA
      E  VOL.(SEGÚN FRECUENCIA)
      F  TIEMPO ESTIMADO POR UNIDAD EN MINUTOS
      G  PROCESO DEL ÁREA
      H  AUTONOMÍA DE LA TAREA (0-100%)      → valor decimal (0.5 = 50%)
      I  TIPO DE ACTIVIDAD (DENTRO DEL PHVA)
      J  VICEPRESIDENCIA
      K  OBSERVACIONES

    El dict devuelto usa como clave el nombre exacto del encabezado
    en mayúsculas para que col_map pueda hacer el match.
    """
    autonomia_raw = act.get("autonomia")
    autonomia_decimal = (autonomia_raw / 100) if autonomia_raw is not None else ""

    return {
        "NO.":                                      numero if numero is not None else "",
        "DESCRIPCIÓN DE LA ACTIVIDAD":              act.get("nombre", ""),
        "UNIDAD DE MEDIDA":                         act.get("unidad_medida", ""),
        "FRECUENCIA":                               act.get("frecuencia", ""),
        "VOL.(SEGÚN FRECUENCIA)":                   act.get("volumen", ""),
        "TIEMPO ESTIMADO POR UNIDAD EN MINUTOS":    act.get("duracion_min", ""),
        "PROCESO DEL ÁREA":                         act.get("proceso_area", ""),
        "AUTONOMÍA DE LA TAREA (0-100%)":           autonomia_decimal,
        "TIPO DE ACTIVIDAD (DENTRO DEL PHVA)":      act.get("phva", ""),
        "VICEPRESIDENCIA":                          vicepresidencia,
        "OBSERVACIONES":                            act.get("observaciones", ""),
    }

#### Parte del Usuario

In [9]:
procesador = ProcesadorTranscripcionTeams()

# En vez del flujo con Excel:
actividades = procesador.procesar_actividades_conversacional()

# Ya están listas para calcular métricas:
metricas = procesador.calcular_metricas(actividades)


════════════════════════════════════════════════════════════
  👤  INFORMACIÓN DEL COLABORADOR
════════════════════════════════════════════════════════════



¿Cuál es tu cargo actual? (Ej: Analista de Operaciones, Coordinador de Proyectos):  Gerente

¿A qué vicepresidencia o área perteneces? (Ej: Tecnología, Operaciones, Finanzas):  Talento y Administrativa



────────────────────────────────────────────────────────────
  Verifica que la información esté correcta:
    Cargo           : Gerente
    Vicepresidencia : Talento y Administrativa
────────────────────────────────────────────────────────────



¿La información es correcta? (si / no):  si



✅ Contexto registrado: Gerente — Talento y Administrativa

════════════════════════════════════════════════════════════
  🗂️  REGISTRO DE ACTIVIDADES — MODO CONVERSACIONAL
════════════════════════════════════════════════════════════

A continuación te haré preguntas sobre cada actividad que realizas.
Puedes registrar todas las que quieras.
Cuando termines, escribe 'no' cuando te pregunte si hay otra actividad.

────────────────────────────────────────────────────────────
  📋  ACTIVIDAD 1
────────────────────────────────────────────────────────────


...


✅ Actividad 3 registrada.

  ⏱️  Carga acumulada: [███░░░░░░░░░░░░░░░░░] 1.4 h de 8.5 h hábiles  (7.1 h disponibles)

────────────────────────────────────────────────────────────


¿Tienes otra actividad para registrar? (si / no):  si{



════════════════════════════════════════════════════════════
  ✅  Se registraron 3 actividad(es) en total.
════════════════════════════════════════════════════════════

🤖 Enriqueciendo actividades con IA...

📋 Generando resumen para tu confirmación final...
📋 RESUMEN PARA CONFIRMACIÓN

Resumen de actividades del colaborador:

1. Actividad: Liderar el diseño y actualización del roadmap de Delivery en Negocio  
- Frecuencia: Semanal  
- Duración: 60 minutos  
- Proceso del área: Consolidación del equipo de Cultura y Transformación  
- Volumen: 1  
- Unidad de medida: reuniones  
- Tipo de actividad (PHVA): Planear  
- Autonomía: 100%  
- Observaciones: Sesión de alineación de prioridades con el equipo de leads de agilidad  

2. Actividad: Liderar el diseño y actualización del roadmap de transformación organizacional  
- Frecuencia: Semanal  
- Duración: 120 minutos  
- Proceso del área: Consolidación del equipo de Cultura y Transformación  
- Volumen: 1  
- Unidad de medida: reuniones  


¿La información es correcta? (si / no):  si



📥 Excel generado: levantamiento_cargas.xlsx


#### Parte del Analista

In [179]:
resumen = procesador.resumen_para_confirmacion(
    actividades_finales_nodiarias,
    contexto
)

In [180]:
metricas = procesador.calcular_metricas(actividades_finales_final)

In [181]:
analisis = procesador.analisis_analista_ia(
        actividades_finales_final,
        metricas,
        contexto
    )

In [42]:
display(Markdown("## 📋 Actividades"))

df_act = pd.DataFrame(actividades_finales_final)
display(df_act)

display(Markdown("## 📊 Métricas por PHVA"))
df_phva = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_phva"],
    "# Act.": metricas["cantidad_actividades_phva"],
    "% Act.": metricas["porcentaje_actividades_phva"]
})
display(df_phva)

display(Markdown("## 📊 Métricas por FRECUENCIA"))
df_frecuencia = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_frecuencia"],
    "# Act.": metricas["cantidad_actividades_frecuencia"],
    "% Act.": metricas["porcentaje_actividades_frecuencia"]
})
display(df_frecuencia)

display(Markdown("## 📊 Métricas por PROCESO"))
df_proceso = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_proceso_area"],
    "# Act.": metricas["cantidad_actividades_proceso_area"],
    "% Act.": metricas["porcentaje_actividades_proceso_area"]
})
display(df_proceso)

display(Markdown("## 📊 TOTALES"))
df_totales = pd.DataFrame([{
    "Total Carga W sin TM": metricas["total_carga_w_sin_tm"],
    "Total Carga Trabajo Individual": metricas["total_carga_trabajo_individual"],
    "Minutos diarios empleados": metricas["minutos_diarios_empleados"],
    "Horas diarias requeridas": metricas["horas_diarias_requeridas"]
}])
display(df_totales)

display(Markdown("## 📊 DOTACIÓN"))
df_dotacion = pd.DataFrame([{
    "Jornada Laboral Diaria": metricas["jornada_laboral_diaria"],
    "Factor Tiempo Neto Productivo": metricas["factor_tiempo_neto_productivo"],
    "Horas Netas Efectivas por Persona": metricas["horas_netas_efectivas_por_persona"],
    "Número Personas Requeridas": metricas["numero_personas_requeridas"]
}])
display(df_dotacion)

display(Markdown("## 📊 AJUSTE FINAL"))
df_ajuste = pd.DataFrame([{
    "Tiempo Muerto": metricas["tiempo_muerto"],
    "Horas Diarias Requeridas Final": metricas["horas_diarias_requeridas_final"]
}])
display(df_ajuste)

display(Markdown("## 🧠 Análisis del Analista IA"))
display(Markdown(analisis))

## 📋 Actividades

,nombre,duracion_min,unidad_medida,phva,frecuencia,volumen,proceso_area,autonomia,observaciones,metricas
0,Revisión de plan de trabajo individual y envío...,30,informes,Verificar,Diario,3,Desarrollo institucional de nuevas formas de t...,100.0,Revision de pendientes y nuevas solicitudes.,"{'minutos_diarios': 90, 'minutos_mes': 1890, '..."
1,Liderar el diseño y actualización del roadmap...,60,reuniones,Planear,Semanal,2,Consolidación del equipo de Cultura y Transfor...,100.0,Sesion de alineación de prioridades con el equ...,"{'minutos_diarios': 24.0, 'minutos_mes': 504.0..."
2,Liderar el diseño y actualización del roadmap...,120,reuniones,Planear,Semanal,1,Consolidación del equipo de Cultura y Transfor...,65.0,Alineación Lideres Dirección Cultura y Transfo...,"{'minutos_diarios': 24.0, 'minutos_mes': 504.0..."
3,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
4,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
5,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
6,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
7,Sesion de alineación de agilidad con aval (adl),60,reuniones,Planear,Quincenal,1,Desarrollo institucional de nuevas formas de t...,20.0,Alineación de formas de trabajo con areas de a...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
8,Agile GO Frente personas,90,seguimientos,Hacer,Semanal,1,Desarrollo institucional de nuevas formas de t...,10.0,Seguimiento a la ejecución de la estrategia co...,"{'minutos_diarios': 18.0, 'minutos_mes': 378.0..."
9,Agile GO Frente empresas,90,seguimientos,Hacer,Semanal,1,Desarrollo institucional de nuevas formas de t...,10.0,Seguimiento a la ejecución de la estrategia c...,"{'minutos_diarios': 18.0, 'minutos_mes': 378.0..."


## 📊 Métricas por PHVA

,Carga de Trabajo,# Act.,% Act.
Verificar,0.1874,6,0.3252
Planear,0.1744,7,0.3026
Hacer,0.2100,4,0.3644
Actuar,0.0045,1,0.0079


## 📊 Métricas por FRECUENCIA

...

,Jornada Laboral Diaria,Factor Tiempo Neto Productivo,Horas Netas Efectivas por Persona,Número Personas Requeridas
0,8.5,0.8676,7.37,0.927


## 📊 AJUSTE FINAL

,Tiempo Muerto,Horas Diarias Requeridas Final
0,0.1324,6.84


## 🧠 Análisis del Analista IA

1. **Nivel de carga laboral (con justificación)**

La carga laboral total calculada para el cargo de Gerente en la Vicepresidencia de Talento y Administrativa es de aproximadamente **6.84 horas diarias requeridas finales**, sobre una jornada laboral estándar de **8.5 horas diarias**. Esto representa un nivel de ocupación del **80.5%** de la jornada laboral disponible (6.84/8.5). La dotación requerida es de **0.927 personas**, lo que indica que la carga está dimensionada para una sola persona, sin necesidad de refuerzos adicionales.

Este nivel de carga puede clasificarse como **ADECUADO**. La carga no es excesiva ni baja; permite un margen para imprevistos, pausas y actividades no planificadas, considerando además un factor de tiempo neto productivo del **86.76%**. La carga diaria en minutos es de **362.29 minutos** (6.04 horas netas antes de ajustar tiempos muertos), lo que es razonable para un cargo gerencial con múltiples responsabilidades estratégicas y operativas.

2. **Validación de coherencia de las métricas**

Las métricas presentadas son coherentes con un cargo gerencial de alta responsabilidad y autonomía. La autonomía promedio declarada es alta, con valores que oscilan entre el **10% y 100%**, predominando actividades con autonomía superior al 50%, lo cual es consistente con la naturaleza del cargo que implica toma de decisiones y liderazgo.

No se observan valores atípicos en duración o volumen que distorsionen la carga total. La distribución de frecuencias muestra predominancia de actividades semanales (62.93% de las actividades y 36.27% de la carga), lo cual es típico en roles gerenciales que requieren seguimiento constante pero no microgestión diaria excesiva. La carga diaria (14.32%) corresponde a una sola actividad de revisión de plan de trabajo, lo que es razonable.

Se detecta una concentración significativa de carga en el proceso "Desarrollo institucional de nuevas formas de trabajo y agilidad" (61.91% de las actividades y 35.68% de la carga), lo que es coherente con la función estratégica del cargo, aunque se debe vigilar que no genere dependencia excesiva en un solo ámbito.

3. **Interpretación del balance PHVA**

La distribución de carga por PHVA es:

- Hacer: 21.0%
- Verificar: 18.74%
- Planear: 17.44%
- Actuar: 0.45%

El rol presenta un balance relativamente equilibrado entre las fases de **Hacer**, **Verificar** y **Planear**, con una ligera preponderancia en la ejecución (Hacer) y verificación. Esto indica que el gerente no solo diseña y planifica estrategias, sino que también participa activamente en el seguimiento y control de las iniciativas.

La fase de **Actuar** está subrepresentada (0.45%), lo que puede reflejar que las acciones correctivas o de mejora continua se delegan o se integran dentro de otras fases. No se observa una sobregestión ni una baja ejecución, pero la baja carga en Actuar podría ser un área a fortalecer para asegurar la mejora continua.

4. **Riesgos operativos identificados**

- **Sobrecarga puntual en actividades de alta frecuencia**: La actividad diaria de revisión de planes y correos consume un 14.32% de la carga, lo que puede generar fatiga si se suma a otras responsabilidades urgentes.
- **Dependencia en el proceso de agilidad**: Más del 60% de las actividades están concentradas en "Desarrollo institucional de nuevas formas de trabajo y agilidad", lo que puede generar cuellos de botella si el gerente no puede atender todas las demandas.
- **Baja autonomía en algunas actividades críticas**: Actividades como "Agile GO Frente personas" y "Sesión de alineación con aval (adl)" tienen autonomías bajas (10% y 20%), lo que puede generar dependencia de otros actores y retrasos.
- **Frecuencias quincenales y trimestrales con baja carga, pero posibles picos de trabajo**: Actividades como steerco y feria Bdb, aunque poco frecuentes, requieren dedicación concentrada que puede afectar la planificación semanal.

5. **Oportunidades de automatización o mejora**

- La actividad diaria de revisión de planes y envío de correos (90 minutos diarios) es repetitiva y con alta frecuencia; podría beneficiarse de herramientas que automaticen alertas o resúmenes para optimizar tiempo.
- Actividades con baja autonomía y alta duración, como las sesiones de alineación y seguimiento (60-90 minutos), podrían mejorar su eficiencia mediante agendas más estructuradas o predefinidas.
- La gestión de iniciativas estratégicas transversales (5 casos semanales, 60 minutos diarios) representa una carga significativa (9.55%) y es 100% autónoma; sin embargo, puede ser un área para estandarizar procesos o delegar parcialmente.
- La formación continua semanal (120 minutos) es necesaria, pero podría optimizarse con formatos flexibles o integrados en la rutina.

6. **Recomendaciones finales**

- Mantener el nivel de carga actual, dado que es adecuado, pero vigilar la concentración en el proceso de agilidad para evitar dependencia excesiva y posibles cuellos de botella.
- Fomentar un mayor énfasis en la fase de Actuar para fortalecer la mejora continua y evitar que las acciones correctivas se diluyan o deleguen sin seguimiento.
- Evaluar la redistribución o delegación de actividades con baja autonomía y alta duración para mejorar la eficiencia operativa y reducir riesgos de dependencia.
- Promover la implementación de mecanismos de automatización o estandarización en actividades de alta frecuencia y repetitividad, especialmente en la revisión diaria de planes y gestión de iniciativas.
- Considerar la flexibilización o integración de la formación continua para optimizar el tiempo sin afectar la actualización profesional.
- Finalmente, asegurar que el gerente cuente con espacios para actividades no planificadas o estratégicas que no están reflejadas en la carga actual, para mantener la capacidad de respuesta y liderazgo efectivo.

In [43]:
dfs = {
    "Actividades": df_act,
    "PHVA": df_phva,
    "Dotacion": df_dotacion,
    "Ajuste": df_ajuste,
    "Frecuencia": df_frecuencia,
    "Totales": df_totales,
    "Proceso_Area": df_proceso
}

In [44]:
ruta_excel = f"analisis_carga_laboral_{contexto['cargo']}.xlsx"

with pd.ExcelWriter(ruta_excel, engine="openpyxl") as writer:
    df_act.to_excel(writer, sheet_name="Actividades", index=True)
    df_phva.to_excel(writer, sheet_name="Métricas PHVA", index=True)
    df_frecuencia.to_excel(writer, sheet_name="Frecuencias", index=True)
    df_proceso.to_excel(writer, sheet_name="Procesos", index=True)
    df_dotacion.to_excel(writer, sheet_name="Dotacion", index=False)
    df_ajuste.to_excel(writer, sheet_name="Ajuste", index=False)
    df_totales.to_excel(writer, sheet_name="Totales", index=False)
    
    lineas_analisis = analisis.split("\n")

    df_analisis = pd.DataFrame({
        "Análisis del Analista IA": lineas_analisis
    })

    df_analisis.to_excel(writer, sheet_name='Analisis IA', index=False)

In [45]:
wb = load_workbook(ruta_excel)

for sheet in wb.sheetnames:
    ws = wb[sheet]
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 25

wb.save(ruta_excel)

In [46]:
display(HTML(f"""
<a href="{ruta_excel}" download>
📥 Descargar archivo Excel de análisis de carga laboral
</a>
"""))

# Guardado en BigQuery